In [9]:
import xarray as xr

ds = xr.open_dataset(r"C:\Users\liyoumin\Desktop\Alfalfa-SPEI\nclimgrid-spei-pearson-03.nc")
print(ds)

<xarray.Dataset> Size: 5GB
Dimensions:  (lat: 596, lon: 1385, time: 1574)
Coordinates:
  * lat      (lat) float32 2kB 24.56 24.6 24.65 24.69 ... 49.27 49.31 49.35
  * lon      (lon) float32 6kB -124.7 -124.6 -124.6 ... -67.1 -67.06 -67.02
  * time     (time) datetime64[ns] 13kB 1895-01-01 1895-02-01 ... 2026-02-01
Data variables:
    spei_03  (time, lat, lon) float32 5GB ...
Attributes: (12/16)
    date_created:              2026-03-04  16:39:37
    date_modified:             2026-03-04  16:39:37
    title:                     SPEI/Pearson 3-month values computed by NIDIS/...
    source:                    NIDIS/NCEI/NOAA
    summary:                   SPEI/Pearson 3-month values computed by NIDIS/...
    institution:               National Centers for Environmental Information...
    ...                        ...
    geospatial_lat_min:        24.5625
    geospatial_lat_max:        49.354168
    geospatial_lon_min:        -124.6875
    geospatial_lon_max:        -67.020836
    geospa

In [10]:
print(ds.data_vars)
print(ds.coords)

Data variables:
    spei_03  (time, lat, lon) float32 5GB ...
Coordinates:
  * lat      (lat) float32 2kB 24.56 24.6 24.65 24.69 ... 49.27 49.31 49.35
  * lon      (lon) float32 6kB -124.7 -124.6 -124.6 ... -67.1 -67.06 -67.02
  * time     (time) datetime64[ns] 13kB 1895-01-01 1895-02-01 ... 2026-02-01


In [23]:
import xarray as xr
import pandas as pd
import os

# Open NetCDF
ds = xr.open_dataset(r"C:\Users\liyoumin\Desktop\Alfalfa-SPEI\nclimgrid-spei-pearson-03.nc")
spei = ds["spei_03"]

# Keep only Apr-Sep months
gs = spei.where(spei.time.dt.month.isin([4, 5, 6, 7, 8, 9]), drop=True)

# Annual Apr-Sep mean
annual_mean = gs.groupby("time.year").mean(dim="time")

# Keep only 2005-2025
annual_mean = annual_mean.sel(year=slice(2005, 2025))

# Output folder
outfolder = r"C:\Users\liyoumin\Desktop\Alfalfa-SPEi\yearly_csv"
os.makedirs(outfolder, exist_ok=True)

# Export one CSV per year
for y in annual_mean.year.values:
    y = int(y)
    print(f"processing {y}...")

    df_y = annual_mean.sel(year=y).to_dataframe(name="spei_apr_sep_mean").reset_index()
    df_y.to_csv(os.path.join(outfolder, f"spei_{y}_apr_sep_pixels.csv"), index=False)

print("done")

processing 2005...
processing 2006...
processing 2007...
processing 2008...
processing 2009...
processing 2010...
processing 2011...
processing 2012...
processing 2013...
processing 2014...
processing 2015...
processing 2016...
processing 2017...
processing 2018...
processing 2019...
processing 2020...
processing 2021...
processing 2022...
processing 2023...
processing 2024...
processing 2025...
done


In [8]:
import arcpy
import os

workdir = r"C:\Users\liyoumin\Desktop\Alfalfa-SPEI"
districts_fc = os.path.join(workdir, r"County_alf_irr\cb_2018_us_cd116_500k\cb_2018_us_cd116_500k.shp")
states_dissolved = os.path.join(workdir, "states_dissolved.shp")

arcpy.management.Dissolve(
    in_features=districts_fc,
    out_feature_class=states_dissolved,
    dissolve_field="STATEFP"
)

print(states_dissolved)

C:\Users\liyoumin\Desktop\Alfalfa-SPEI\states_dissolved.shp


In [23]:
import xarray as xr
import numpy as np
import arcpy
from arcpy.sa import *
import os

arcpy.CheckOutExtension("Spatial")
arcpy.ResetEnvironments()
arcpy.env.overwriteOutput = True

# ---------------------------
# paths
# ---------------------------
workdir = r"C:\Users\liyoumin\Desktop\Alfalfa-SPEI"
spei_nc = os.path.join(workdir, "nclimgrid-spei-pearson-03.nc")
cdl_tif = os.path.join(workdir, r"2022_30m_cdls\2008_30m_cdls.tif")

# use dissolved state boundary shapefile here
# if you only have congressional districts, dissolve by STATEFP first and use that output
states_fc = os.path.join(workdir, r"states_dissolved.shp")
zone_field = "STATEFP"

alfalfa_code = 36

# ---------------------------
# read inputs once
# ---------------------------
ds = xr.open_dataset(spei_nc)
spei = ds["spei_03"]

cdl = Raster(cdl_tif)
cdl_sr = arcpy.Describe(cdl).spatialReference
cdl_cell = arcpy.Describe(cdl).meanCellWidth

print("SPEI exists:", arcpy.Exists(spei_nc))
print("CDL exists:", arcpy.Exists(cdl_tif))
print("States exists:", arcpy.Exists(states_fc))

for year in range(2005, 2008):
    print(f"Processing {year}...")

    # -------------------------
    # 1) annual Apr-Sep mean SPEI
    # -------------------------
    spei_y = spei.where(
        (spei.time.dt.year == year) & (spei.time.dt.month.isin([4, 5, 6, 7, 8, 9])),
        drop=True
    )

    spei_y_mean = spei_y.mean(dim="time")

    # clear env before NumPyArrayToRaster
    arcpy.env.snapRaster = None
    arcpy.env.extent = None
    arcpy.env.cellSize = None

    arr = spei_y_mean.values.astype(np.float32)
    arr_flip = np.flipud(arr)

    lon = spei_y_mean["lon"].values
    lat = spei_y_mean["lat"].values

    xmin = float(lon.min())
    xmax = float(lon.max())
    ymin = float(lat.min())
    ymax = float(lat.max())

    nrows, ncols = arr.shape
    x_cell = (xmax - xmin) / (ncols - 1)
    y_cell = (ymax - ymin) / (nrows - 1)

    lower_left = arcpy.Point(xmin - x_cell / 2, ymin - y_cell / 2)

    temp_raster = arcpy.NumPyArrayToRaster(
        arr_flip,
        lower_left_corner=lower_left,
        x_cell_size=x_cell,
        y_cell_size=y_cell,
        value_to_nodata=np.nan
    )

    # save only temporary rasters/tables, then delete them
    temp_spei_tif = os.path.join(workdir, f"_tmp_spei_{year}.tif")
    temp_spei_30m_tif = os.path.join(workdir, f"_tmp_spei_{year}_30m.tif")
    temp_alfalfa_tif = os.path.join(workdir, f"_tmp_alfalfa_spei_{year}.tif")
    temp_table = os.path.join(workdir, f"_tmp_zonal_{year}.dbf")
    out_csv = os.path.join(workdir, f"alfal_spei_mean_{year}.csv")

    arcpy.management.DefineProjection(temp_raster, arcpy.SpatialReference(4326))
    temp_raster.save(temp_spei_tif)

    # -------------------------
    # 2) project/match to CDL
    # -------------------------
    arcpy.env.snapRaster = cdl
    arcpy.env.cellSize = cdl
    arcpy.env.extent = cdl.extent

    arcpy.management.ProjectRaster(
        in_raster=temp_spei_tif,
        out_raster=temp_spei_30m_tif,
        out_coor_system=cdl_sr,
        resampling_type="NEAREST",
        cell_size=cdl_cell
    )

    # -------------------------
    # 3) mask by alfalfa
    # -------------------------
    spei30 = Raster(temp_spei_30m_tif)
    alfalfa_spei = SetNull(cdl != alfalfa_code, spei30)
    alfalfa_spei.save(temp_alfalfa_tif)

    # -------------------------
    # 4) zonal stats to table
    # -------------------------
    ZonalStatisticsAsTable(
        in_zone_data=states_fc,
        zone_field=zone_field,
        in_value_raster=temp_alfalfa_tif,
        out_table=temp_table,
        ignore_nodata="DATA",
        statistics_type="MEAN"
    )

    # -------------------------
    # 5) write yearly CSV only
    # -------------------------
    fields = [zone_field, "MEAN", "COUNT"]
    with open(out_csv, "w", encoding="utf-8") as f:
        f.write("year,statefp,mean_spei,count\n")
        with arcpy.da.SearchCursor(temp_table, fields) as cursor:
            for row in cursor:
                f.write(f"{year},{row[0]},{row[1]},{row[2]}\n")

    print("Saved:", out_csv)

    # -------------------------
    # 6) delete temp raster/table
    # -------------------------
    for tmp in [temp_spei_tif, temp_spei_30m_tif, temp_alfalfa_tif, temp_table]:
        if arcpy.Exists(tmp):
            arcpy.management.Delete(tmp)

print("Done.")

SPEI exists: True
CDL exists: True
States exists: True
Processing 2005...
Saved: C:\Users\liyoumin\Desktop\Alfalfa-SPEI\alfal_spei_mean_2005.csv
Done.


In [27]:
import pandas as pd
import os
import glob

workdir = r"C:\Users\liyoumin\Desktop\Alfalfa-SPEI\alf_spei_05_25_aprset"

# state FIPS -> postal abbreviation
fips_to_stusps = {
    "01":"AL","02":"AK","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE",
    "11":"DC","12":"FL","13":"GA","15":"HI","16":"ID","17":"IL","18":"IN","19":"IA",
    "20":"KS","21":"KY","22":"LA","23":"ME","24":"MD","25":"MA","26":"MI","27":"MN",
    "28":"MS","29":"MO","30":"MT","31":"NE","32":"NV","33":"NH","34":"NJ","35":"NM",
    "36":"NY","37":"NC","38":"ND","39":"OH","40":"OK","41":"OR","42":"PA","44":"RI",
    "45":"SC","46":"SD","47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA",
    "54":"WV","55":"WI","56":"WY"
}

all_files = sorted(glob.glob(os.path.join(workdir, "alfal_spei_mean_*.csv")))

dfs = []
for f in all_files:
    df = pd.read_csv(f, dtype={"statefp": str})
    
    # keep only needed columns
    df = df[["year", "statefp", "mean_spei"]].copy()
    
    # pad statefp with leading zero if needed
    df["statefp"] = df["statefp"].str.zfill(2)
    
    # add postal abbreviation
    df["stusps"] = df["statefp"].map(fips_to_stusps)
    
    # rename mean variable
    df = df.rename(columns={"mean_spei": "speimean"})
    
    # reorder columns
    df = df[["statefp", "stusps", "year", "speimean"]]
    
    dfs.append(df)

panel = pd.concat(dfs, ignore_index=True)

# optional: sort
panel = panel.sort_values(["statefp", "year"]).reset_index(drop=True)

out_csv = os.path.join(workdir, "alfal_spei_panel_2005_2025.csv")
panel.to_csv(out_csv, index=False)

print("Saved:", out_csv)
print(panel.head(20))
print(panel.shape)

Saved: C:\Users\liyoumin\Desktop\Alfalfa-SPEI\alf_spei_05_25_aprset\alfal_spei_panel_2005_2025.csv
   statefp stusps  year  speimean
0       01     AL  2005  0.096219
1       01     AL  2006 -1.100994
2       01     AL  2007 -1.810719
3       01     AL  2008 -0.307458
4       01     AL  2009  0.906669
5       01     AL  2010 -0.736350
6       01     AL  2011  0.505627
7       01     AL  2012 -0.740764
8       01     AL  2013  0.788690
9       01     AL  2014  0.028956
10      01     AL  2015  0.082793
11      01     AL  2016 -1.016290
12      01     AL  2017  0.416010
13      01     AL  2018  0.161167
14      01     AL  2019 -0.613796
15      01     AL  2020  0.946254
16      01     AL  2021  1.194477
17      01     AL  2022  0.708246
18      01     AL  2023  0.002722
19      01     AL  2024 -0.046525
(998, 4)
